In [ ]:
import numpy as np
import pandas as pd

# Import block is un-sorted or un-formatted
from sklearn.datasets import load_iris
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from students.razin.lesson2 import Exercise

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

# Создаем бинарную задачу: setosa (класс 0), остальные (1 и 2)
y_binary = (y == 0).astype(int)  # 1 если Setosa, 0 если нет

print(f"Размер данных: {X.shape}")
print(f"Setosa: {sum(y_binary == 1)} шт, Другие: {sum(y_binary == 0)} шт")
print(f"Названия признаков: {iris.feature_names}")

In [ ]:
# отделяем train (60%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_binary,
    test_size=0.4,  # 40% во временную выборку
    random_state=42,
    stratify=y_binary,
    shuffle=True,
)

# временную выборку делим на validation и test - по 20% каждая
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp, shuffle=True
)

print(f"Обучающая выборка (train): {len(X_train)} образцов ({len(X_train) / len(X) * 100:.0f}%)")
print(f"Валидационная выборка (val): {len(X_val)} образцов ({len(X_val) / len(X) * 100:.0f}%)")
print(f"Тестовая выборка (test): {len(X_test)} образцов ({len(X_test) / len(X) * 100:.0f}%)")

print("\nРаспределение классов:")
print(f"Train: Setosa={sum(y_train == 1)}, Другие={sum(y_train == 0)}")
print(f"Val:   Setosa={sum(y_val == 1)}, Другие={sum(y_val == 0)}")
print(f"Test:  Setosa={sum(y_test == 1)}, Другие={sum(y_test == 0)}")

In [ ]:
# Вычисляем параметры нормализации на train
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# Нормируем все выборки одними и теми же параметрами
X_train_norm = (X_train - mean) / std
X_val_norm = (X_val - mean) / std
X_test_norm = (X_test - mean) / std

print("Параметры нормализации (по train):")
print(f"Средние: {mean}")
print(f"Стандартные отклонения: {std}")

In [ ]:
def evaluate_model(lr, batch_size, X_train, X_val, y_train, y_val, n_epochs=25, seed=42):
    # создаем модель и обучаем ее
    model = Exercise.create_logistic_model(num_features=4, rng=np.random.default_rng(seed))
    Exercise.fit(model, X_train, y_train, lr=lr, n_epoch=n_epochs, batch_size=batch_size)

    # Оцениваем на validation
    y_pred_proba = model.predict(X_val)
    y_pred = (y_pred_proba > 0.5).astype(int)

    # смотрим процент совпадений
    accuracy = np.mean(y_pred == y_val)

    auroc = roc_auc_score(y_val, y_pred_proba)

    return accuracy, auroc, model

In [ ]:
# Подбор гиперпараметров с проверкой на разных seed
learning_rates = [
    0.001,
    0.003,
    0.005,
    0.007,
    0.009,
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.06,
    0.07,
    0.08,
    0.09,
    0.1,
    0.12,
    0.14,
    0.16,
    0.18,
    0.2,
    0.22,
    0.24,
    0.26,
    0.28,
    0.3,
    0.35,
    0.4,
    0.45,
    0.5,
    0.55,
    0.6,
    0.65,
    0.7,
    0.75,
    0.8,
    0.85,
    0.9,
    0.95,
    1.0,
]
batch_sizes = [4, 8, 16, 32, 64, None]
seeds = [1, 2, 3, 4, 5]

results = []

for lr in learning_rates:
    for bs in batch_sizes:
        auroc_scores = []
        accuracy_scores = []

        for seed in seeds:
            accuracy, auroc, _model = evaluate_model(
                lr, bs, X_train_norm, X_val_norm, y_train, y_val, n_epochs=25, seed=seed
            )
            auroc_scores.append(auroc)
            accuracy_scores.append(accuracy)

        results.append(
            {
                "lr": lr,
                "batch_size": bs,
                "mean_auroc": np.mean(auroc_scores),
                "std_auroc": np.std(auroc_scores),
                "mean_accuracy": np.mean(accuracy_scores),
                "std_accuracy": np.std(accuracy_scores),
                "min_auroc": np.min(auroc_scores),
                "max_auroc": np.max(auroc_scores),
                "min_accuracy": np.min(accuracy_scores),
                "max_accuracy": np.max(accuracy_scores),
                "auroc_scores": auroc_scores,
                "accuracy_scores": accuracy_scores,
            }
        )

df_results = pd.DataFrame(results)

df_results["stability_auroc"] = df_results["mean_auroc"] - df_results["std_auroc"]
df_results["stability_accuracy"] = df_results["mean_accuracy"] - df_results["std_accuracy"]

# ТОП ПО AUROC
df_results_auroc = df_results.sort_values("stability_auroc", ascending=False)
print("ТОП ПО AUROC")
print("=" * 70)
print(
    df_results_auroc[["lr", "batch_size", "mean_auroc", "std_auroc", "stability_auroc", "mean_accuracy"]]
    .head(5)
    .to_string()
)

# ТОП ПО ACCURACY
df_results_accuracy = df_results.sort_values("stability_accuracy", ascending=False)
print("\n" + "=" * 70)
print("ТОП ПО ACCURACY")
print("=" * 70)
print(
    df_results_accuracy[["lr", "batch_size", "mean_accuracy", "std_accuracy", "stability_accuracy", "mean_auroc"]]
    .head(5)
    .to_string()
)

# Лучшие по AUROC
best_auroc_row = df_results_auroc.iloc[0]

print("\n" + "=" * 70)
print("Лучшие параметры (по AUROC):")
print(f"lr = {best_auroc_row['lr']}")
print(f"batch_size = {best_auroc_row['batch_size']}")
print(
    f"""mean AUROC = {best_auroc_row["mean_auroc"]:.4f} ± {best_auroc_row["std_auroc"]:.4f} 
    (stability={best_auroc_row["stability_auroc"]:.4f})"""
)

# Лучшие по Accuracy
best_accuracy_row = df_results_accuracy.iloc[0]

print("\n" + "=" * 70)
print("Лучшие параметры (по ACCURACY):")
print(f"lr = {best_accuracy_row['lr']}")
print(f"batch_size = {best_accuracy_row['batch_size']}")
print(
    f"""mean ACCURACY = {best_accuracy_row["mean_accuracy"]:.4f} ± {best_accuracy_row["std_accuracy"]:.4f} 
    (stability={best_accuracy_row["stability_accuracy"]:.4f})"""
)